# 🐍 Django Intermediate Interview Prep

A complete deep-dive reference covering:

| # | Topic |
|---|-------|
| 1 | ORM & Queries |
| 2 | REST Framework (DRF) |
| 3 | Models & Migrations |
| 4 | Views & URLs |
| 5 | Authentication & Permissions |
| 6 | Signals & Middleware |
| 7 | Forms & Validation |

---
# 🗄️ 1. ORM & Queries

## Q1. `select_related()` vs `prefetch_related()`

In [ ]:
# select_related() — for ForeignKey / OneToOne (SQL JOIN)

# ❌ Without select_related — causes N+1 queries
books = Book.objects.all()
for book in books:
    print(book.author.name)  # hits DB every iteration!

# ✅ With select_related — single SQL JOIN query
books = Book.objects.select_related('author').all()
for book in books:
    print(book.author.name)  # no extra DB hit

# Generated SQL:
# SELECT book.*, author.* FROM book
# INNER JOIN author ON book.author_id = author.id;

# ---

# prefetch_related() — for ManyToMany / reverse FK (separate query + Python join)

# ❌ Without prefetch_related — N+1 problem
authors = Author.objects.all()
for author in authors:
    print(author.books.all())  # separate query per author!

# ✅ With prefetch_related — 2 queries total, joined in Python
authors = Author.objects.prefetch_related('books').all()
for author in authors:
    print(author.books.all())  # no extra DB hit

# Generated SQL:
# SELECT * FROM author;
# SELECT * FROM book WHERE author_id IN (1, 2, 3, ...);

# Quick Rule:
# select_related  → ForeignKey, OneToOne  → JOIN          → joined in DB
# prefetch_related → ManyToMany, reverse FK → Separate query → joined in Python

## Q2. `Q` Objects — AND / OR / NOT

In [ ]:
from django.db.models import Q

# Simple AND (same as filter(status='active', category='books'))
Book.objects.filter(Q(status='active') & Q(category='books'))

# OR
Book.objects.filter(Q(status='active') | Q(status='pending'))

# NOT
Book.objects.filter(~Q(status='deleted'))

# Complex: (active OR pending) AND category=books
Book.objects.filter(
    (Q(status='active') | Q(status='pending')) & Q(category='books')
)

# Dynamically building Q objects
filters = Q()
if some_condition:
    filters &= Q(status='active')
if another_condition:
    filters |= Q(category='books')

Book.objects.filter(filters)

## Q3. N+1 Query Problem

In [ ]:
# ❌ BAD — 1 query for orders + 1 query per order for user = N+1
orders = Order.objects.all()        # Query 1
for order in orders:
    print(order.user.email)         # Query 2, 3, 4 ... N+1

# ✅ GOOD — just 1 JOIN query
orders = Order.objects.select_related('user').all()
for order in orders:
    print(order.user.email)         # no extra queries

# ✅ For ManyToMany
orders = Order.objects.prefetch_related('items').all()
for order in orders:
    print(order.items.all())        # no extra queries

# How to detect N+1:
from django.db import connection
print(len(connection.queries))  # count total queries

## Q4. `values()` vs `values_list()`

In [ ]:
# Normal queryset — returns model instances
users = User.objects.all()
# <QuerySet [<User: john>, <User: jane>]>

# values() — returns list of dicts
users = User.objects.values('id', 'username')
# <QuerySet [{'id': 1, 'username': 'john'}, {'id': 2, 'username': 'jane'}]>

# values_list() — returns list of tuples
users = User.objects.values_list('id', 'username')
# <QuerySet [(1, 'john'), (2, 'jane')]>

# values_list with flat=True — returns flat list (single field only)
usernames = User.objects.values_list('username', flat=True)
# <QuerySet ['john', 'jane']>

# When to use:
# values()              → dict-like access, serializing to JSON
# values_list(flat=True) → simple list of IDs or single field
# Both are faster than full model instances (skip model instantiation)

## Q5. `annotate()` vs `aggregate()`

In [ ]:
from django.db.models import Count, Avg, Sum

# aggregate() — returns a SINGLE dict for the entire queryset
result = Order.objects.aggregate(total=Sum('amount'), avg=Avg('amount'))
# {'total': 5000, 'avg': 250.0}

# annotate() — adds a computed field to EACH object in the queryset
authors = Author.objects.annotate(book_count=Count('books'))
for author in authors:
    print(author.name, author.book_count)  # per-author count

# Filter on annotated fields
prolific = Author.objects.annotate(
    book_count=Count('books')
).filter(book_count__gte=3)

# aggregate() → SELECT SUM(...)         → single dict
# annotate()  → SELECT ..., COUNT(...)  → queryset with extra field per row

## Q6. `F()` Expression

In [1]:
from django.db.models import F, ExpressionWrapper, FloatField

# ❌ BAD — fetches value into Python, then updates (race condition risk!)
product = Product.objects.get(id=1)
product.stock = product.stock - 1
product.save()

# ✅ GOOD — atomic update entirely in DB
Product.objects.filter(id=1).update(stock=F('stock') - 1)

# Compare two fields on the same model
Product.objects.filter(discount_price__lt=F('original_price'))

# Use in annotations
Product.objects.annotate(
    savings=ExpressionWrapper(
        F('original_price') - F('discount_price'),
        output_field=FloatField()
    )
)

# Why F() matters:
# - Avoids race conditions in concurrent environments
# - More efficient — no round trip to Python
# - Works great with update(), filter(), and annotate()

NameError: name 'Product' is not defined

---
# 🔌 2. REST Framework (DRF)

## Q12. `Serializer` vs `ModelSerializer`

In [ ]:
from rest_framework import serializers

# Serializer — Manual, full control
class UserSerializer(serializers.Serializer):
    id = serializers.IntegerField(read_only=True)
    username = serializers.CharField(max_length=150)
    email = serializers.EmailField()

    def create(self, validated_data):
        return User.objects.create(**validated_data)

    def update(self, instance, validated_data):
        instance.username = validated_data.get('username', instance.username)
        instance.email = validated_data.get('email', instance.email)
        instance.save()
        return instance


# ModelSerializer — Auto-generates fields from model
class UserSerializer(serializers.ModelSerializer):
    class Meta:
        model = User
        fields = ['id', 'username', 'email']
        read_only_fields = ['id']
        extra_kwargs = {
            'password': {'write_only': True}
        }
    # create() and update() are auto-implemented!

# Key Differences:
# Serializer      → Fields defined manually, create/update manual, non-model data
# ModelSerializer → Fields auto-generated, create/update auto, model-backed APIs

## Q13. DRF Authentication vs Django's Built-in Authentication

In [ ]:
# Django Built-in — Session based (for browser)
from django.contrib.auth import authenticate, login

user = authenticate(request, username='john', password='secret')
if user:
    login(request)  # sets session cookie

# ---

# DRF Authentication — Token/JWT based (for APIs)
# settings.py
REST_FRAMEWORK = {
    'DEFAULT_AUTHENTICATION_CLASSES': [
        'rest_framework.authentication.TokenAuthentication',
        'rest_framework_simplejwt.authentication.JWTAuthentication',
        'rest_framework.authentication.SessionAuthentication',
    ]
}

# DRF Built-in Token Auth — urls.py
from rest_framework.authtoken.views import obtain_auth_token
# path('api/token/', obtain_auth_token)
# Client sends: Authorization: Token abc123xyz

# JWT Auth — urls.py
from rest_framework_simplejwt.views import TokenObtainPairView, TokenRefreshView
# path('api/token/', TokenObtainPairView.as_view())
# path('api/token/refresh/', TokenRefreshView.as_view())
# Client sends: Authorization: Bearer <access_token>

# Comparison:
# Session  → Server-side session DB  → Browser apps
# Token    → DB table                → Simple APIs
# JWT      → Stateless (no DB)       → Scalable / mobile APIs

## Q14. `APIView` vs `ViewSet` vs `ModelViewSet`

In [ ]:
from rest_framework.views import APIView
from rest_framework.response import Response
from rest_framework import viewsets
from rest_framework.decorators import action
from rest_framework.routers import DefaultRouter

# APIView — Most control, manual routing
class UserView(APIView):
    def get(self, request, pk=None):
        user = User.objects.get(pk=pk)
        return Response(UserSerializer(user).data)

    def post(self, request):
        serializer = UserSerializer(data=request.data)
        if serializer.is_valid():
            serializer.save()
            return Response(serializer.data, status=201)
        return Response(serializer.errors, status=400)


# ViewSet — Groups related views, needs router
class UserViewSet(viewsets.ViewSet):
    def list(self, request):         # GET /users/
        pass
    def retrieve(self, request, pk): # GET /users/1/
        pass

    @action(detail=True, methods=['post'])
    def activate(self, request, pk): # POST /users/1/activate/
        pass


# ModelViewSet — Full CRUD auto-wired
class UserViewSet(viewsets.ModelViewSet):
    queryset = User.objects.all()
    serializer_class = UserSerializer
    # Automatically provides:
    # GET    /users/   → list()
    # POST   /users/   → create()
    # GET    /users/1/ → retrieve()
    # PUT    /users/1/ → update()
    # PATCH  /users/1/ → partial_update()
    # DELETE /users/1/ → destroy()

# Auto routing
router = DefaultRouter()
router.register('users', UserViewSet)
urlpatterns = router.urls

# When to use:
# APIView      → Custom logic, non-CRUD endpoints
# ViewSet      → Grouped views with some custom actions
# ModelViewSet → Standard CRUD on a model with minimal code

## Q15. Custom Permissions in DRF

In [ ]:
from rest_framework.permissions import BasePermission, SAFE_METHODS

# Owner-only permission
class IsOwner(BasePermission):
    def has_permission(self, request, view):
        # Request-level — is user authenticated?
        return request.user and request.user.is_authenticated

    def has_object_permission(self, request, view, obj):
        # Object-level — is this user the owner?
        return obj.owner == request.user


# Read-only for unauthenticated, full access for authenticated
class IsAuthenticatedOrReadOnly(BasePermission):
    def has_permission(self, request, view):
        if request.method in SAFE_METHODS:  # GET, HEAD, OPTIONS
            return True
        return request.user.is_authenticated


# Apply to ViewSet
class PostViewSet(viewsets.ModelViewSet):
    permission_classes = [IsOwner]
    # OR combine (AND logic — all must pass)
    # permission_classes = [IsAuthenticated, IsOwner]


# has_permission     → called first, request level (auth checks, role checks)
# has_object_permission → called on retrieve/update/destroy (ownership checks)

## Q16. `get_queryset()` vs `queryset` directly

In [ ]:
# ❌ Static queryset — evaluated once, same for all users
class PostViewSet(viewsets.ModelViewSet):
    queryset = Post.objects.all()  # fixed, no per-request logic possible
    serializer_class = PostSerializer


# ✅ get_queryset() — dynamic, per-request filtering
class PostViewSet(viewsets.ModelViewSet):
    serializer_class = PostSerializer

    def get_queryset(self):
        # Show only the logged-in user's posts
        return Post.objects.filter(author=self.request.user)


# Dynamic — filter based on URL query params
class PostViewSet(viewsets.ModelViewSet):
    serializer_class = PostSerializer

    def get_queryset(self):
        queryset = Post.objects.all()
        status = self.request.query_params.get('status')
        if status:
            queryset = queryset.filter(status=status)
        return queryset

# Rule: Always use get_queryset() when the queryset depends
# on the request, user, or URL parameters.

## Q17. Nested Serializers for Related Models

In [ ]:
# Read-only Nested Serializer
class AuthorSerializer(serializers.ModelSerializer):
    class Meta:
        model = Author
        fields = ['id', 'name']

class BookSerializer(serializers.ModelSerializer):
    author = AuthorSerializer(read_only=True)  # nested

    class Meta:
        model = Book
        fields = ['id', 'title', 'author']


# Writable Nested Serializer (override create/update)
class BookSerializer(serializers.ModelSerializer):
    author = AuthorSerializer()

    class Meta:
        model = Book
        fields = ['id', 'title', 'author']

    def create(self, validated_data):
        author_data = validated_data.pop('author')
        author, _ = Author.objects.get_or_create(**author_data)
        return Book.objects.create(author=author, **validated_data)

    def update(self, instance, validated_data):
        author_data = validated_data.pop('author', None)
        if author_data:
            author, _ = Author.objects.get_or_create(**author_data)
            instance.author = author
        return super().update(instance, validated_data)


# Nested ManyToMany (many=True)
class BookSerializer(serializers.ModelSerializer):
    tags = TagSerializer(many=True, read_only=True)

    class Meta:
        model = Book
        fields = ['id', 'title', 'tags']


# Lighter alternative — PrimaryKeyRelatedField
class BookSerializer(serializers.ModelSerializer):
    author = serializers.PrimaryKeyRelatedField(queryset=Author.objects.all())

    class Meta:
        model = Book
        fields = ['id', 'title', 'author']

---
# 🏗️ 3. Models & Migrations

## Q18. `null=True` vs `blank=True`

In [ ]:
from django.db import models

class UserProfile(models.Model):
    # null=True only → DB allows NULL, but form validation still requires a value
    bio = models.TextField(null=True)

    # blank=True only → form allows empty string, DB stores '' not NULL
    nickname = models.CharField(max_length=50, blank=True)

    # Both → DB allows NULL and form validation allows empty (most common for optional)
    website = models.URLField(null=True, blank=True)

    # Neither (default) → required in both DB and forms
    username = models.CharField(max_length=150)

# When to use:
# CharField, TextField         → blank=True only (store '' not NULL)
# IntegerField, DateField, FK  → null=True, blank=True for optional
# BooleanField                 → null=True if 3-state needed

# ❌ Avoid null=True on string fields — leads to two empty states: NULL and ''
# name = models.CharField(max_length=100, null=True)

# ✅ Use blank=True for optional string fields
# name = models.CharField(max_length=100, blank=True, default='')

## Q19. `makemigrations` vs `migrate`

In [ ]:
# makemigrations → detects model changes, creates migration FILES (no DB touch)
# python manage.py makemigrations

# migrate → applies migration files to the DATABASE (alters DB schema)
# python manage.py migrate

# Flow:
# Model Changes → makemigrations → 0002_xxx.py → migrate → SQL ALTER TABLE

# Useful commands:
# python manage.py showmigrations                    → see pending migrations
# python manage.py sqlmigrate myapp 0002             → preview SQL (no execute)
# python manage.py migrate myapp 0001                → rollback to 0001
# python manage.py makemigrations myapp --empty      → empty migration (for data)
# python manage.py migrate myapp 0002 --fake         → mark applied without running

print("Run these in terminal, not in Python!")

## Q20. Handling Migration Conflicts in a Team

In [ ]:
# Conflict scenario:
# 0001_initial
#     ├── 0002_add_bio      (developer A)
#     └── 0002_add_nickname (developer B) ← conflict!

# Step 1 — Detect conflict:
# python manage.py makemigrations
# CommandError: Conflicting migrations detected: 0002_add_bio, 0002_add_nickname

# Step 2 — Merge:
# python manage.py makemigrations --merge
# Creates: 0003_merge_0002_add_bio_0002_add_nickname.py

# The merge file looks like:
class Migration(migrations.Migration):
    dependencies = [
        ('myapp', '0002_add_bio'),
        ('myapp', '0002_add_nickname'),  # both are now dependencies
    ]
    operations = []  # no operations, just resolves the branch

# Step 3 — Apply:
# python manage.py migrate

# Best practices:
# ✅ Communicate with teammates before creating migrations
# ✅ Pull latest code before running makemigrations
# ✅ Keep migrations small and focused
# ✅ Never edit applied migration files

## Q21. `through` Model in ManyToManyField

In [ ]:
# Without through — simple M2M, no extra data
class Student(models.Model):
    courses = models.ManyToManyField('Course')

# With through — store extra info about the relationship
class Student(models.Model):
    courses = models.ManyToManyField('Course', through='Enrollment')

class Course(models.Model):
    name = models.CharField(max_length=100)

class Enrollment(models.Model):  # the through model
    student = models.ForeignKey(Student, on_delete=models.CASCADE)
    course = models.ForeignKey(Course, on_delete=models.CASCADE)
    # Extra fields on the relationship itself
    enrolled_on = models.DateField(auto_now_add=True)
    grade = models.CharField(max_length=2, blank=True)
    is_active = models.BooleanField(default=True)

    class Meta:
        unique_together = ('student', 'course')  # prevent duplicate enrollments


# Querying
# Create enrollment explicitly (can't use .add() with through models)
Enrollment.objects.create(student=student, course=course, grade='A')

# Access through the M2M relation (still works)
student.courses.all()  # returns Course queryset

# Access extra data via the through model
Enrollment.objects.filter(student=student, grade='A')

## Q22. `Meta` Class Options

In [ ]:
class Article(models.Model):
    title = models.CharField(max_length=200)
    author = models.ForeignKey(User, on_delete=models.CASCADE)
    category = models.CharField(max_length=50)
    slug = models.SlugField()
    created_at = models.DateTimeField(auto_now_add=True)
    views = models.IntegerField(default=0)

    class Meta:
        ordering = ['-created_at']                          # default ordering
        unique_together = [['author', 'slug']]              # composite unique
        indexes = [
            models.Index(fields=['category', 'created_at']),# composite index
            models.Index(fields=['slug']),                  # single field index
        ]
        db_table = 'blog_articles'                          # custom table name
        verbose_name = 'Article'
        verbose_name_plural = 'Articles'
        permissions = [
            ('can_publish', 'Can publish articles'),
        ]
        # Django 4.x+ — more powerful constraints
        constraints = [
            models.UniqueConstraint(
                fields=['author', 'slug'],
                name='unique_author_slug'
            ),
            models.CheckConstraint(
                check=models.Q(views__gte=0),
                name='views_non_negative'
            )
        ]

# Override Meta ordering
Article.objects.all().order_by('title')  # overrides Meta.ordering
Article.objects.all().order_by()         # removes all ordering

## Q23. `on_delete` Options for ForeignKey

In [ ]:
class Post(models.Model):
    # CASCADE   → delete post when author is deleted
    author = models.ForeignKey(User, on_delete=models.CASCADE)

    # SET_NULL  → set to NULL when category is deleted
    category = models.ForeignKey(Category, on_delete=models.SET_NULL, null=True)

    # SET_DEFAULT → set to default value when editor is deleted
    editor = models.ForeignKey(
        User, on_delete=models.SET_DEFAULT, default=1, related_name='edited_posts'
    )

    # PROTECT   → prevent deleting User if they have posts
    reviewer = models.ForeignKey(
        User, on_delete=models.PROTECT, related_name='reviewed_posts'
    )

    # DO_NOTHING → dangerous, can cause IntegrityError
    legacy_ref = models.ForeignKey(OldModel, on_delete=models.DO_NOTHING)


# All on_delete options:
# CASCADE     → Deletes the child object too
# SET_NULL    → Sets FK to NULL (requires null=True)
# SET_DEFAULT → Sets FK to default value
# SET(value)  → Sets FK to a specific value or callable
# PROTECT     → Raises ProtectedError — prevents deletion
# RESTRICT    → Like PROTECT but allows deletion if other CASCADE handles it
# DO_NOTHING  → No DB action — can cause IntegrityError

# SET with a callable
def get_default_author():
    return User.objects.get(username='admin').id

class Post(models.Model):
    author = models.ForeignKey(User, on_delete=models.SET(get_default_author))

---
# 🌐 4. Views & URLs

## Q7. FBV vs CBV

In [ ]:
from django.shortcuts import render, get_object_or_404
from django.views.generic import ListView, DetailView, CreateView, UpdateView, DeleteView
from django.contrib.auth.decorators import login_required
from django.utils.decorators import method_decorator

# FBV — Simple, explicit
def post_list(request):
    posts = Post.objects.all()
    return render(request, 'posts/list.html', {'posts': posts})

def post_detail(request, pk):
    post = get_object_or_404(Post, pk=pk)
    if request.method == 'GET':
        return render(request, 'posts/detail.html', {'post': post})


# CBV — Reusable, structured
class PostListView(ListView):
    model = Post
    template_name = 'posts/list.html'
    context_object_name = 'posts'
    paginate_by = 10

class PostDetailView(DetailView):
    model = Post
    template_name = 'posts/detail.html'

class PostCreateView(CreateView):
    model = Post
    fields = ['title', 'content', 'category']
    success_url = '/posts/'


# Applying decorators in CBV
@method_decorator(login_required, name='dispatch')
class PostCreateView(CreateView):
    pass

# FBV equivalent — simpler
@login_required
def post_create(request):
    pass

# When to use:
# FBV → Simple, custom logic, easier to debug
# CBV → Reusable code, standard CRUD, inheritance + mixins

## Q8. `get_context_data()` in CBVs

In [ ]:
from django.db.models import Count

class PostDetailView(DetailView):
    model = Post
    template_name = 'posts/detail.html'

    def get_context_data(self, **kwargs):
        context = super().get_context_data(**kwargs)  # always call super() first!

        # Add extra data to context
        context['related_posts'] = Post.objects.filter(
            category=self.object.category
        ).exclude(pk=self.object.pk)[:5]

        context['comment_count'] = self.object.comments.count()
        context['is_owner'] = self.object.author == self.request.user

        return context


class PostListView(ListView):
    model = Post

    def get_context_data(self, **kwargs):
        context = super().get_context_data(**kwargs)
        context['categories'] = Category.objects.annotate(post_count=Count('posts'))
        context['total_posts'] = Post.objects.count()
        return context


# get_queryset()      → controls WHAT objects are listed
# get_context_data()  → controls WHAT ELSE goes into the template

## Q9. Mixins in Django Views

In [ ]:
from django.contrib.auth.mixins import LoginRequiredMixin, PermissionRequiredMixin, UserPassesTestMixin

# LoginRequiredMixin — always FIRST in inheritance
class PostCreateView(LoginRequiredMixin, CreateView):
    model = Post
    fields = ['title', 'content']
    login_url = '/login/'

# PermissionRequiredMixin
class PostDeleteView(PermissionRequiredMixin, DeleteView):
    model = Post
    permission_required = 'blog.delete_post'

# UserPassesTestMixin — custom logic
class PostUpdateView(UserPassesTestMixin, UpdateView):
    model = Post
    def test_func(self):
        return self.request.user == self.get_object().author


# Custom Mixins
class SetAuthorMixin:
    def form_valid(self, form):
        form.instance.author = self.request.user
        return super().form_valid(form)

class SidebarMixin:
    def get_context_data(self, **kwargs):
        context = super().get_context_data(**kwargs)
        context['categories'] = Category.objects.all()
        context['recent_posts'] = Post.objects.order_by('-created_at')[:5]
        return context

# Combine multiple mixins — MRO goes left to right
class PostCreateView(LoginRequiredMixin, SetAuthorMixin, SidebarMixin, CreateView):
    model = Post
    fields = ['title', 'content']
    # LoginRequiredMixin checked first in dispatch()

## Q10. Passing Extra Data via `path()`

In [ ]:
from django.urls import path, register_converter

# Basic URL with path converter
# path('posts/<int:pk>/', PostDetailView.as_view(), name='post-detail')

# Passing extra static kwargs
# path('posts/featured/', PostListView.as_view(), {'status': 'featured'}, name='featured')

# Built-in path converters:
# <int:pk>      → integer: 1, 2, 3
# <str:slug>    → string (no slash): hello-world
# <slug:slug>   → slug: letters, numbers, hyphens
# <uuid:id>     → UUID: 550e8400-e29b...
# <path:filepath> → path with slashes: docs/2024/report


# Custom converter
class FourDigitYearConverter:
    regex = '[0-9]{4}'

    def to_python(self, value):
        return int(value)

    def to_url(self, value):
        return '%04d' % value

register_converter(FourDigitYearConverter, 'yyyy')
# path('archive/<yyyy:year>/', ArchiveView.as_view())


# Receiving extra kwargs in CBV
class PostListView(ListView):
    model = Post

    def get(self, request, *args, **kwargs):
        self.status = self.kwargs.get('status')  # from path() extra kwargs
        return super().get(request, *args, **kwargs)

    def get_queryset(self):
        if self.status:
            return Post.objects.filter(status=self.status)
        return Post.objects.all()

## Q11. `reverse()` and `resolve()`

In [ ]:
from django.urls import reverse, resolve, reverse_lazy
from django.shortcuts import redirect

# reverse() — URL name → URL string
url = reverse('post-detail', kwargs={'pk': 5})
# Returns: '/posts/5/'

# With query params (manual)
from urllib.parse import urlencode
base = reverse('post-list')
query = urlencode({'category': 'tech', 'page': 2})
url = f'{base}?{query}'
# Returns: '/posts/?category=tech&page=2'

# Redirect using reverse
# return redirect(reverse('post-detail', kwargs={'pk': post.pk}))
# return redirect('post-detail', pk=post.pk)  # shortcut


# reverse_lazy() — for class-level attributes (evaluated lazily)
class PostCreateView(CreateView):
    model = Post
    success_url = reverse_lazy('post-list')  # ✅ works
    # success_url = reverse('post-list')     # ❌ NoReverseMatch error!


# resolve() — URL string → view function/name
match = resolve('/posts/5/')
print(match.func)       # <function PostDetailView>
print(match.view_name)  # 'post-detail'
print(match.kwargs)     # {'pk': 5}
print(match.app_name)   # 'blog'

# reverse() → URL name  → URL string  (generate URLs in code)
# resolve() → URL string → view info  (inspect what a URL points to)

---
# 🔐 5. Authentication & Permissions

## Q24. Authentication vs Authorization

In [ ]:
from django.contrib.auth import authenticate, login
from django.core.exceptions import PermissionDenied

# Authentication — WHO are you? (Identity)
def login_view(request):
    user = authenticate(
        request,
        username=request.POST['username'],
        password=request.POST['password']
    )
    if user is not None:
        login(request)  # sets session
        return redirect('dashboard')
    return render(request, 'login.html', {'error': 'Invalid credentials'})


# Authorization — WHAT can you do? (Access Control)
def delete_post(request, pk):
    post = get_object_or_404(Post, pk=pk)
    if post.author != request.user and not request.user.is_staff:
        raise PermissionDenied  # 403 Forbidden
    post.delete()
    return redirect('post-list')


# Django's Authorization Layers:
# Layer 1 — is_authenticated
# if request.user.is_authenticated: ...

# Layer 2 — is_staff / is_superuser
# if request.user.is_staff: ...

# Layer 3 — Model-level permissions
# if request.user.has_perm('blog.delete_post'): ...

# Layer 4 — Object-level permissions
# if request.user.has_perm('blog.delete_post', post_object): ...

## Q25. Session-Based Authentication Under the Hood

In [ ]:
# Flow:
# 1. User submits login form
# 2. authenticate() → verifies credentials against DB
# 3. login(request, user)
# 4. Django creates Session in DB (django_session table)
# 5. Response: Set-Cookie: sessionid=abc123; HttpOnly; Path=/
# 6. Browser sends cookie with every request
# 7. SessionMiddleware reads cookie → fetches user → sets request.user

# Required middleware (settings.py):
MIDDLEWARE = [
    'django.contrib.sessions.middleware.SessionMiddleware',
    'django.contrib.auth.middleware.AuthenticationMiddleware',
]

# Session storage options:
SESSION_ENGINE = 'django.contrib.sessions.backends.db'            # default (DB)
SESSION_ENGINE = 'django.contrib.sessions.backends.cache'         # faster (Redis)
SESSION_ENGINE = 'django.contrib.sessions.backends.signed_cookies' # cookie-based

# Session settings:
SESSION_COOKIE_AGE = 1209600        # 2 weeks
SESSION_COOKIE_SECURE = True        # HTTPS only
SESSION_COOKIE_HTTPONLY = True      # JS can't access

# Manually working with sessions
def my_view(request):
    request.session['cart_items'] = [1, 2, 3]       # store
    cart = request.session.get('cart_items', [])    # read
    del request.session['cart_items']               # delete key
    request.session.flush()                         # clear entire session

## Q26. JWT Authentication in DRF

In [ ]:
# pip install djangorestframework-simplejwt

# settings.py
from datetime import timedelta

REST_FRAMEWORK = {
    'DEFAULT_AUTHENTICATION_CLASSES': [
        'rest_framework_simplejwt.authentication.JWTAuthentication',
    ]
}

SIMPLE_JWT = {
    'ACCESS_TOKEN_LIFETIME': timedelta(minutes=15),
    'REFRESH_TOKEN_LIFETIME': timedelta(days=7),
    'ROTATE_REFRESH_TOKENS': True,
    'BLACKLIST_AFTER_ROTATION': True,
    'ALGORITHM': 'HS256',
    'AUTH_HEADER_TYPES': ('Bearer',),
}

# urls.py
# path('api/token/', TokenObtainPairView.as_view())         → login
# path('api/token/refresh/', TokenRefreshView.as_view())    → refresh
# path('api/token/verify/', TokenVerifyView.as_view())      → verify

# JWT Flow:
# 1. POST /api/token/ {username, password}
# 2. Server returns {access: ..., refresh: ...}
# 3. Client sends: Authorization: Bearer <access_token>
# 4. Access token expires → send refresh token to /api/token/refresh/
# 5. Server returns new access token


# Customizing JWT payload
from rest_framework_simplejwt.serializers import TokenObtainPairSerializer
from rest_framework_simplejwt.views import TokenObtainPairView

class MyTokenObtainPairSerializer(TokenObtainPairSerializer):
    @classmethod
    def get_token(cls, user):
        token = super().get_token(user)
        token['username'] = user.username
        token['email'] = user.email
        token['is_staff'] = user.is_staff
        return token

class MyTokenObtainPairView(TokenObtainPairView):
    serializer_class = MyTokenObtainPairSerializer

# Session vs JWT:
# Session → Server-side DB, easy logout, browser apps
# JWT     → Stateless, scales easily, APIs/mobile/microservices

## Q27. DRF Permission Classes

In [ ]:
from rest_framework.permissions import AllowAny, IsAuthenticated, IsAdminUser

# Global default (settings.py)
REST_FRAMEWORK = {
    'DEFAULT_PERMISSION_CLASSES': ['rest_framework.permissions.IsAuthenticated']
}

# Per view
class PublicPostList(APIView):
    permission_classes = [AllowAny]

class AdminDashboard(APIView):
    permission_classes = [IsAuthenticated, IsAdminUser]  # AND logic


# Per action in ViewSet
class PostViewSet(viewsets.ModelViewSet):

    def get_permissions(self):
        if self.action in ['list', 'retrieve']:
            return [AllowAny()]
        elif self.action in ['create']:
            return [IsAuthenticated()]
        else:  # update, destroy
            return [IsAuthenticated(), IsOwner()]

    @action(detail=True, methods=['post'], permission_classes=[IsAdminUser])
    def feature(self, request, pk):
        pass

# Built-in classes:
# AllowAny                  → anyone, including unauthenticated
# IsAuthenticated           → must be logged in
# IsAdminUser               → must be is_staff=True
# IsAuthenticatedOrReadOnly → read for all, write for authenticated

## Q28. Restricting Access to Object Owners Only

In [ ]:
from rest_framework.permissions import BasePermission, SAFE_METHODS

# Method 1 — Custom Permission Class (recommended)
class IsOwnerOrReadOnly(BasePermission):
    def has_object_permission(self, request, view, obj):
        if request.method in SAFE_METHODS:  # GET, HEAD, OPTIONS
            return True
        return obj.author == request.user

class PostViewSet(viewsets.ModelViewSet):
    permission_classes = [IsAuthenticated, IsOwnerOrReadOnly]


# Method 2 — Override perform_update / perform_destroy
class PostViewSet(viewsets.ModelViewSet):
    permission_classes = [IsAuthenticated]

    def perform_update(self, serializer):
        if self.get_object().author != self.request.user:
            raise PermissionDenied("You can only edit your own posts.")
        serializer.save()


# Method 3 — Filter queryset to own objects only
class PostViewSet(viewsets.ModelViewSet):
    permission_classes = [IsAuthenticated]

    def get_queryset(self):
        return Post.objects.filter(author=self.request.user)

    def perform_create(self, serializer):
        serializer.save(author=self.request.user)


# Best practice — combine all three
class PostViewSet(viewsets.ModelViewSet):
    serializer_class = PostSerializer
    permission_classes = [IsAuthenticated, IsOwnerOrReadOnly]

    def get_queryset(self):
        return Post.objects.filter(author=self.request.user)

    def perform_create(self, serializer):
        serializer.save(author=self.request.user)

---
# 📡 6. Signals & Middleware

## Q29. Django Signals — `pre_save` and `post_save`

In [ ]:
from django.db.models.signals import pre_save, post_save, pre_delete, post_delete
from django.dispatch import receiver

# pre_save — runs BEFORE saving to DB (good for modifying data)
@receiver(pre_save, sender=Post)
def pre_save_post(sender, instance, **kwargs):
    if not instance.slug:
        from django.utils.text import slugify
        instance.slug = slugify(instance.title)

    if instance.status == 'published' and not instance.published_at:
        from django.utils import timezone
        instance.published_at = timezone.now()


# post_save — runs AFTER saving to DB
@receiver(post_save, sender=User)
def post_save_user(sender, instance, created, **kwargs):
    if created:                                          # new object
        UserProfile.objects.create(user=instance)
        # send_welcome_email.delay(instance.email)       # Celery task
    else:                                                # existing object updated
        instance.profile.save()


# Register signals in apps.py
from django.apps import AppConfig

class BlogConfig(AppConfig):
    name = 'blog'

    def ready(self):
        import blog.signals  # triggers all @receiver decorators

# settings.py — use BlogConfig not just 'blog'
# INSTALLED_APPS = ['blog.apps.BlogConfig']

## Q30. Signals vs Overriding `save()`

In [ ]:
# Overriding save() — tightly coupled, lives in the model
class Post(models.Model):
    title = models.CharField(max_length=200)
    slug = models.SlugField(blank=True)

    def save(self, *args, **kwargs):
        if not self.slug:
            from django.utils.text import slugify
            self.slug = slugify(self.title)
        super().save(*args, **kwargs)  # always call super()!


# Signal — decoupled, lives outside the model
@receiver(post_save, sender=Post)
def handle_post_save(sender, instance, created, **kwargs):
    if created:
        notify_subscribers(instance)


# When to use which:
# save() override → logic belongs to the model (slug, timestamps)
# Signal          → logic belongs to another app (notifications, cache)
# Signal          → need created vs updated distinction
# Signal          → triggering side effects (email, tasks)
# Signal          → keeping apps decoupled
# save() override → simple, single-app logic

## Q31. Preventing Infinite Loops in `post_save`

In [ ]:
# ❌ INFINITE LOOP — post_save calls save() which triggers post_save again!
@receiver(post_save, sender=UserProfile)
def sync_profile(sender, instance, **kwargs):
    instance.updated_at = timezone.now()
    instance.save()  # ← triggers post_save again → infinite loop!


# Solution 1 — Use update() on queryset (doesn't trigger signals)
@receiver(post_save, sender=UserProfile)
def sync_profile(sender, instance, created, **kwargs):
    if not created:
        UserProfile.objects.filter(pk=instance.pk).update(
            updated_at=timezone.now()
        )


# Solution 2 — Disconnect before save, reconnect after
@receiver(post_save, sender=UserProfile)
def sync_profile(sender, instance, **kwargs):
    post_save.disconnect(sync_profile, sender=UserProfile)
    instance.save()
    post_save.connect(sync_profile, sender=UserProfile)


# Solution 3 — Flag on the instance
@receiver(post_save, sender=UserProfile)
def sync_profile(sender, instance, **kwargs):
    if getattr(instance, '_syncing', False):
        return
    instance._syncing = True
    instance.save()
    instance._syncing = False

## Q32. Django Middleware — Request/Response Cycle

In [ ]:
# Request flows DOWN through middleware, response flows UP in reverse:
# SecurityMiddleware → SessionMiddleware → AuthenticationMiddleware → View
# View → AuthenticationMiddleware → SessionMiddleware → SecurityMiddleware

# Built-in middleware explained:
MIDDLEWARE = [
    'django.middleware.security.SecurityMiddleware',        # HTTPS, HSTS, headers
    'django.contrib.sessions.middleware.SessionMiddleware', # reads/writes session
    'django.middleware.common.CommonMiddleware',            # www/slash redirects
    'django.middleware.csrf.CsrfViewMiddleware',            # CSRF token validation
    'django.contrib.auth.middleware.AuthenticationMiddleware', # sets request.user
    'django.contrib.messages.middleware.MessageMiddleware', # flash messages
    'django.middleware.clickjacking.XFrameOptionsMiddleware', # X-Frame-Options
]

# Middleware structure — new style (callable class)
class MyMiddleware:
    def __init__(self, get_response):
        self.get_response = get_response
        # One-time setup on server start

    def __call__(self, request):
        # Code runs BEFORE the view

        response = self.get_response(request)  # calls next middleware/view

        # Code runs AFTER the view

        return response

# Middleware order rules:
# SecurityMiddleware          → always FIRST
# SessionMiddleware           → before AuthenticationMiddleware
# AuthenticationMiddleware    → after SessionMiddleware
# Custom middleware           → after Auth if you need request.user

## Q33. Custom Middleware — Request Logger

In [ ]:
import time
import logging
import uuid

logger = logging.getLogger(__name__)

# Request logger middleware
class RequestLoggerMiddleware:
    def __init__(self, get_response):
        self.get_response = get_response

    def __call__(self, request):
        start_time = time.time()
        logger.info(f"[REQUEST] {request.method} {request.path} user={request.user}")

        response = self.get_response(request)

        duration = time.time() - start_time
        logger.info(
            f"[RESPONSE] {request.method} {request.path} "
            f"status={response.status_code} duration={duration:.2f}s"
        )
        return response


# Maintenance mode middleware
class MaintenanceModeMiddleware:
    def __init__(self, get_response):
        self.get_response = get_response

    def __call__(self, request):
        from django.conf import settings
        from django.http import HttpResponse

        if getattr(settings, 'MAINTENANCE_MODE', False):
            if not request.user.is_staff:
                return HttpResponse('Site under maintenance.', status=503)

        return self.get_response(request)


# Custom response header middleware
class CustomHeaderMiddleware:
    def __init__(self, get_response):
        self.get_response = get_response

    def __call__(self, request):
        response = self.get_response(request)
        response['X-App-Version'] = '2.1.0'
        response['X-Request-Id'] = str(uuid.uuid4())
        return response

# Register in settings.py:
# MIDDLEWARE = [..., 'myapp.middleware.RequestLoggerMiddleware', ...]

---
# 📝 7. Forms & Validation

## Q34. `Form` vs `ModelForm`

In [ ]:
from django import forms

# Form — Manual, not tied to a model
class ContactForm(forms.Form):
    name = forms.CharField(max_length=100)
    email = forms.EmailField()
    message = forms.CharField(widget=forms.Textarea)
    priority = forms.ChoiceField(choices=[
        ('low', 'Low'), ('medium', 'Medium'), ('high', 'High')
    ])

def contact_view(request):
    if request.method == 'POST':
        form = ContactForm(request.POST)
        if form.is_valid():
            send_email(form.cleaned_data['email'], form.cleaned_data['message'])
            return redirect('success')
    else:
        form = ContactForm()
    return render(request, 'contact.html', {'form': form})


# ModelForm — Auto-generated from a model
class PostForm(forms.ModelForm):
    class Meta:
        model = Post
        fields = ['title', 'content', 'category', 'status']
        exclude = ['author', 'created_at', 'slug']
        widgets = {
            'content': forms.Textarea(attrs={'rows': 10}),
        }
        labels = {'content': 'Post Body'}
        error_messages = {
            'title': {'required': 'Please enter a title.'}
        }

def create_post(request):
    if request.method == 'POST':
        form = PostForm(request.POST)
        if form.is_valid():
            post = form.save(commit=False)  # don't save yet
            post.author = request.user      # set author manually
            post.save()                     # now save
            return redirect('post-detail', pk=post.pk)

# Differences:
# Form      → Fields manual, no .save(), custom only, non-model data
# ModelForm → Fields auto, built-in .save(), inherits model validators

## Q35. `clean_<fieldname>()` vs `clean()`

In [ ]:
class RegisterForm(forms.Form):
    username = forms.CharField(max_length=150)
    email = forms.EmailField()
    age = forms.IntegerField()
    password = forms.CharField(widget=forms.PasswordInput)
    password_confirm = forms.CharField(widget=forms.PasswordInput)

    # clean_<fieldname>() — validates a SINGLE field
    def clean_username(self):
        username = self.cleaned_data['username']
        if ' ' in username:
            raise forms.ValidationError("Username cannot contain spaces.")
        if User.objects.filter(username=username).exists():
            raise forms.ValidationError("This username is already taken.")
        return username  # always return the value!

    def clean_age(self):
        age = self.cleaned_data['age']
        if age < 18:
            raise forms.ValidationError("You must be at least 18 years old.")
        return age

    def clean_email(self):
        email = self.cleaned_data['email'].lower()  # normalize
        if User.objects.filter(email=email).exists():
            raise forms.ValidationError("This email is already registered.")
        return email

    # clean() — validates MULTIPLE fields together
    def clean(self):
        cleaned_data = super().clean()  # always call super() first!
        password = cleaned_data.get('password')
        password_confirm = cleaned_data.get('password_confirm')

        if password and password_confirm:
            if password != password_confirm:
                raise forms.ValidationError("Passwords do not match.")

        # Add error to a specific field
        start_date = cleaned_data.get('start_date')
        end_date = cleaned_data.get('end_date')
        if start_date and end_date and end_date < start_date:
            self.add_error('end_date', 'End date must be after start date.')

        return cleaned_data  # always return cleaned_data!

# Validation order:
# 1. Field.to_python()      → type conversion
# 2. Field.validate()       → built-in validators
# 3. Field.run_validators() → validators= kwarg
# 4. clean_<fieldname>()   → custom single-field
# 5. Form.clean()          → custom cross-field

## Q36. File Uploads with Django Forms

In [ ]:
import os

# Model setup
class Document(models.Model):
    title = models.CharField(max_length=200)
    file = models.FileField(upload_to='documents/%Y/%m/')
    image = models.ImageField(upload_to='images/', blank=True, null=True)

# settings.py
# MEDIA_URL = '/media/'
# MEDIA_ROOT = BASE_DIR / 'media'

# urls.py (dev only)
# urlpatterns += static(settings.MEDIA_URL, document_root=settings.MEDIA_ROOT)


# Form with file validation
class DocumentForm(forms.ModelForm):
    class Meta:
        model = Document
        fields = ['title', 'file', 'image']

    def clean_file(self):
        file = self.cleaned_data.get('file')
        if file:
            if file.size > 5 * 1024 * 1024:  # 5MB
                raise forms.ValidationError("File size must be under 5MB.")
            allowed_extensions = ['.pdf', '.doc', '.docx']
            ext = os.path.splitext(file.name)[1].lower()
            if ext not in allowed_extensions:
                raise forms.ValidationError(
                    f"Only {', '.join(allowed_extensions)} files allowed."
                )
        return file


# View — must pass request.FILES!
def upload_document(request):
    if request.method == 'POST':
        form = DocumentForm(request.POST, request.FILES)  # ← request.FILES required!
        if form.is_valid():
            form.save()
            return redirect('document-list')
    else:
        form = DocumentForm()
    return render(request, 'upload.html', {'form': form})

# Template MUST have enctype:
# <form method="POST" enctype="multipart/form-data">

## Q37. What `form.is_valid()` Does Under the Hood

In [ ]:
# form.is_valid() internally calls:
# def is_valid(self):
#     return self.is_bound and not self.errors

# self.errors triggers full_clean() → runs entire validation chain:
# def full_clean(self):
#     self._clean_fields()   → validates each field + calls clean_<field>()
#     self._clean_form()     → calls self.clean()
#     self._post_clean()     → ModelForm: calls Model.full_clean()

# Step 1 — _clean_fields()
# For each field: convert raw → Python type, run validators, call clean_<field>()

# Step 2 — _clean_form()
# Calls self.clean() — cross-field validation
# ValidationErrors go into self._errors['__all__']

# Step 3 — _post_clean() (ModelForm only)
# Calls Model.full_clean() which runs:
#   Model.clean_fields()    → field-level model validation
#   Model.clean()           → model-level custom validation
#   Model.validate_unique() → unique/unique_together constraints


# Inspecting errors
form = PostForm(request.POST)
if not form.is_valid():
    print(form.errors)
    # {'title': ['This field is required.'], 'content': ['Too short.']}

    print(form.errors.as_json())
    # {"title": [{"message": "This field is required.", "code": "required"}]}

    print(form.non_field_errors())
    # Errors from clean() not tied to a specific field

## Q38. Multi-Field Custom Validation

In [ ]:
from django.core.exceptions import ValidationError

# In a Form
class BookingForm(forms.Form):
    check_in = forms.DateField()
    check_out = forms.DateField()
    guests = forms.IntegerField(min_value=1)
    room_type = forms.ChoiceField(choices=[
        ('single', 'Single'), ('double', 'Double'), ('suite', 'Suite')
    ])

    def clean(self):
        cleaned_data = super().clean()
        check_in = cleaned_data.get('check_in')
        check_out = cleaned_data.get('check_out')
        guests = cleaned_data.get('guests')
        room_type = cleaned_data.get('room_type')

        if check_in and check_out:
            if check_out <= check_in:
                raise forms.ValidationError("Check-out must be after check-in.")
            from datetime import timedelta
            if (check_out - check_in).days > 30:
                raise forms.ValidationError("Maximum stay is 30 days.")

        if guests and room_type:
            max_guests = {'single': 1, 'double': 2, 'suite': 4}
            if guests > max_guests.get(room_type, 1):
                self.add_error('guests',
                    f"A {room_type} room supports max {max_guests[room_type]} guests."
                )
        return cleaned_data


# Custom reusable validators
def validate_no_profanity(value):
    banned_words = ['spam', 'fake']
    for word in banned_words:
        if word in value.lower():
            raise ValidationError(f"Content contains banned word: {word}")

def validate_future_date(value):
    from django.utils import timezone
    if value < timezone.now().date():
        raise ValidationError("Date must be in the future.")

# Use on model field
class Event(models.Model):
    name = models.CharField(max_length=200, validators=[validate_no_profanity])
    date = models.DateField(validators=[validate_future_date])

# Use on form field
class EventForm(forms.Form):
    name = forms.CharField(validators=[validate_no_profanity])
    date = forms.DateField(validators=[validate_future_date])

---
# 🎉 Summary

| # | Topic | Key Questions |
|---|-------|---------------|
| ✅ 1 | ORM & Queries | `select_related`, `Q objects`, N+1, `values()`, `annotate()`, `F()` |
| ✅ 2 | REST Framework | Serializers, Auth, APIView vs ViewSet, Permissions, Nested Serializers |
| ✅ 3 | Models & Migrations | `null/blank`, migrations, conflicts, `through`, `Meta`, `on_delete` |
| ✅ 4 | Views & URLs | FBV vs CBV, `get_context_data`, Mixins, `path()`, `reverse/resolve` |
| ✅ 5 | Auth & Permissions | Session auth, JWT, Permission classes, Owner restriction |
| ✅ 6 | Signals & Middleware | `pre/post_save`, Signal vs `save()`, infinite loops, middleware cycle |
| ✅ 7 | Forms & Validation | `Form vs ModelForm`, `clean()`, file uploads, `is_valid()`, multi-field validation |